# 01 · Data Cleaning & Validation
**Acquisition Campaign Measurement & ROI Analytics Framework**

This notebook loads the raw acquisition datasets, validates data quality, resolves referential integrity issues, and produces the cleaned analysis-ready tables consumed by all downstream notebooks.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from src.data_loader import load_raw_data
from src.preprocessing import DataPreprocessor

pd.set_option('display.max_columns', 50)

## 1. Load Raw Data

In [2]:
data = load_raw_data()
for name, df in data.items():
    print(f'{name:22s} shape={df.shape}')

campaigns              shape=(5200, 17)
customers              shape=(5400, 12)
transactions           shape=(21500, 8)
monthly_performance    shape=(245, 13)


## 2. Data Quality Checks
We check for nulls, duplicate keys, and referential-integrity violations (transactions referencing non-existent campaigns/customers).

In [3]:
for name, df in data.items():
    nulls = df.isnull().sum().sum()
    dupes = df.duplicated().sum()
    print(f'{name}: nulls={nulls}, duplicate_rows={dupes}')

campaigns: nulls=0, duplicate_rows=0
customers: nulls=0, duplicate_rows=0
transactions: nulls=0, duplicate_rows=0
monthly_performance: nulls=0, duplicate_rows=0


In [4]:
# Referential integrity check
orphan_customers = ~data['transactions']['customer_id'].isin(data['customers']['customer_id'])
orphan_campaigns = ~data['transactions']['campaign_id'].isin(data['campaigns']['campaign_id'])
print('Orphan customer refs:', orphan_customers.sum())
print('Orphan campaign refs:', orphan_campaigns.sum())

Orphan customer refs: 0
Orphan campaign refs: 0


## 3. Clean & Standardize
Using `DataPreprocessor` to: remove impossible values (clicks > impressions), clip outlier ROI/ROAS, parse dates, and engineer duration / recency fields.

In [5]:
pre = DataPreprocessor(data)
clean = pre.run_all()
for name, df in clean.items():
    print(f'{name:12s} -> {df.shape}')

2026-07-12 19:49:36,010 | INFO | src.preprocessing | clean_campaigns: removed 0 invalid rows (0.00%)


2026-07-12 19:49:36,052 | INFO | src.preprocessing | build_master_table: 21500 rows, 19 columns


campaigns    -> (5200, 18)
customers    -> (5400, 13)
transactions -> (21500, 9)
monthly_performance -> (245, 13)
master       -> (21500, 19)


In [6]:
clean['campaigns'].describe(include='all').T.head(20)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
campaign_id,5200,5200,CMP000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
campaign_name,5200,1994,Google_ContentSyndication_202301,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
channel,5200,7,Google Ads,1427,NaN,NaN,NaN,NaN,NaN,NaN,NaN
campaign_type,5200,10,Product Launch,542,NaN,NaN,NaN,NaN,NaN,NaN,NaN
start_date,5200,NaN,NaN,NaN,2024-06-10 11:32:51.692307,2023-01-01 00:00:00,2023-09-15 00:00:00,2024-06-11 12:00:00,2025-03-02 00:00:00,2025-11-30 00:00:00,NaN
end_date,5200,NaN,NaN,NaN,2024-07-13 15:42:55.384615,2023-01-11 00:00:00,2023-10-19 00:00:00,2024-07-12 00:00:00,2025-04-05 12:00:00,2026-01-26 00:00:00,NaN
budget,5200.0,NaN,NaN,NaN,5160.338104,51.73,1050.815,3526.68,8549.7425,19878.23,4682.911901
impressions,5200.0,NaN,NaN,NaN,253671.258462,2009.0,126457.0,254559.0,378794.5,499971.0,143937.279498
clicks,5200.0,NaN,NaN,NaN,11169.799423,11.0,3653.5,8421.5,15831.5,62471.0,9805.339252
ctr,5200.0,NaN,NaN,NaN,0.044143,0.0044,0.0244,0.0388,0.059425,0.1356,0.026005


## 4. Persist Cleaned Tables

In [7]:
out_dir = '../data/processed'
import os
os.makedirs(out_dir, exist_ok=True)
for name in ['campaigns','customers','transactions']:
    clean[name].to_csv(f'{out_dir}/{name}_clean.csv', index=False)
clean['master'].to_csv(f'{out_dir}/master_table.csv', index=False)
print('Cleaned tables written to', out_dir)

Cleaned tables written to ../data/processed


### Summary
- No referential integrity issues found (synthetic data generated with FK consistency).
- Outlier ROI/ROAS values winsorized at the 0.5th/99.5th percentile to stabilize downstream modeling.
- A unified `master_table` was created by joining transactions → campaigns → customers, used throughout the segmentation and statistical-testing notebooks.